In [0]:
# ── Config ─────────────────────────────────────────────────────────────
TABLE_NAME = "nifty_live.session_ticks"

# ── Load Data ──────────────────────────────────────────────────────────
df = spark.read.table(TABLE_NAME)

# ── Show Latest Data ───────────────────────────────────────────────────
display(
    df.orderBy("datetime_ist", ascending=False)
      .limit(50)
)

# ── Basic Stats ────────────────────────────────────────────────────────
total_rows = df.count()
display(spark.createDataFrame([(total_rows,)], ["Total rows"]))

latest_time = df.agg({"datetime_ist": "max"}).collect()[0][0]
display(spark.createDataFrame([(latest_time,)], ["Latest timestamp"]))

In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

window = Window.orderBy("datetime_ist")

df_feat = df.withColumn(
    "returns",
    (col("close") - lag("close").over(window)) / lag("close").over(window)
)

display(
    df_feat.orderBy("datetime_ist", ascending=False).limit(50)
)